### LLM & RAG Powered QA bot to enable Q&A session on Indian Penal Code

In [2]:
#Import the necessary libraries
import gradio as gr
import numpy as np
import os,requests

from langchain_community.vectorstores import Chroma
from dotenv import load_dotenv
from langchain_community.document_loaders import PyPDFLoader
from langchain_community.embeddings import HuggingFaceEmbeddings
from groq import Groq
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_text_splitters import CharacterTextSplitter
from langchain_community.embeddings import HuggingFaceEmbeddings

### Define the LLM Model

In [3]:
from langchain_groq import ChatGroq

load_dotenv()

# Use ChatGroq instead of custom class
llm = ChatGroq(
    model=os.getenv("GROQ_MODEL", "mixtral-8x7b-32768"),
    temperature=0.5,
    max_tokens=4096,
    api_key=os.getenv("GROQ_API_KEY")
)

### Setup the Embedding Model

In [4]:
embed_model = HuggingFaceEmbeddings(
    model_name= os.getenv("HF_EMBEDDING_MODEL")
)

embed_model

C:\Users\adars\AppData\Local\Temp\ipykernel_29576\3676308838.py:1: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  embed_model = HuggingFaceEmbeddings(


HuggingFaceEmbeddings(client=SentenceTransformer(
  (0): Transformer({'transformer_task': 'feature-extraction', 'modality_config': {'text': {'method': 'forward', 'method_output_name': 'last_hidden_state'}}, 'module_output_name': 'token_embeddings', 'architecture': 'MPNetModel'})
  (1): Pooling({'embedding_dimension': 768, 'pooling_mode': 'mean', 'include_prompt': True})
  (2): Normalize({})
), model_name='sentence-transformers/all-mpnet-base-v2', cache_folder=None, model_kwargs={}, encode_kwargs={}, multi_process=False, show_progress=False)

### Load Indian Penal Code document to be as context grounding information

In [5]:
# Load the PDF text

pdf_url = "https://www.ncib.in/pdf/indian-penal-code.pdf"
pdf_path = "indian-penal-code.pdf"

if not os.path.exists(pdf_path):
    response = requests.get(pdf_url)
    if response.status_code == 200:
    # Save the PDF to a file
        with open("indian-penal-code.pdf", "wb") as file:
            file.write(response.content)
            print("PDF downloaded successfully!")

    else:
        print(f"Failed to download. Status code: {response.status_code}")
    
    
loader = PyPDFLoader(file_path=pdf_path)

pages = loader.load()

print(f"Sample page content: {pages[0].page_content[:100]}")

print(f"No.of Pages: {len(pages)} ")

Sample page content: Indian Penal Code 1860
Section 1. Title and extent of operation of the Code
Act No. 45 of 1860.
This
No.of Pages: 268 


### Split Document and create the vector embeddings in Vectorstore

In [17]:
# Perform splitting based on Recursive Character Splitter

# Custom separators prioritized for legal documents
ipc_separators = [
    "Section ",
    " section ",
    "\n\n",  # Double newlines often indicate new sections
    "\n",    # Single newline for finer splits
]

splitter = RecursiveCharacterTextSplitter(
    separators=ipc_separators,
    chunk_size=1000,       # Target chunk size (characters)
    chunk_overlap=200,     # Overlap to maintain context
    length_function=len
)

chunks = splitter.split_documents(pages)
print(f'Total chunks: {len(chunks)}')


# Create the vector database

# Re-index
import shutil
shutil.rmtree('./chroma_db', ignore_errors=True)

vectorstore = Chroma.from_documents(
    documents=chunks,
    embedding=embed_model,
    persist_directory='./chroma_db',
    collection_name='collection_'+ np.random.randint(0,9999).__str__()
)

"""
retriever = vectorstore.as_retriever(
    search_type='similarity',
    search_kwargs={'k': 4}
)
"""

print(f'Indexed ✅ — {vectorstore._collection.count()} chunks in vector store')


Total chunks: 843
Indexed ✅ — 843 chunks in vector store


In [19]:
from langchain_community.retrievers import BM25Retriever
from langchain_classic.retrievers import EnsembleRetriever
from langchain_cohere import CohereRerank
from langchain_classic.retrievers import ContextualCompressionRetriever

# BM25 for keyword matching
bm25 = BM25Retriever.from_documents(chunks, k=4)

# Semantic for meaning
semantic = vectorstore.as_retriever(search_kwargs={'k': 4})

# Combine — BM25 handles exact matches, semantic handles intent
retriever = EnsembleRetriever(
    retrievers=[bm25, semantic],
    weights=[0.8, 0.2]  # 80% keyword, 20% semantic
)

In [20]:
def rerank_local(docs, query):
    scored = []
    for doc in docs:
        score = sum(1 for word in query.split() if word.lower() in doc.page_content.lower())
        scored.append((score, doc))
    # Sort by score (first element of tuple)
    return [doc for score, doc in sorted(scored, key=lambda x: x[0], reverse=True)]

### Setup RAG using langchain

In [21]:
# Cell 8 — Conversational RAG chain

from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnableLambda
from langchain_core.output_parsers import StrOutputParser

def rag_pipeline(question:str):

    docs = retriever.invoke(question)

    docs = rerank_local(docs, question)
    
    print("RETRIEVED CHUNKS:")
    for i, doc in enumerate(docs):
        print(f"[{i}] {doc.page_content[:150]}\n")
        
    prompt = ChatPromptTemplate.from_messages([
        ('system', """You are a paralegal. Answer ONLY from context.\nContext: {context}\nQuestion: {question}. 
         DON'T use any prior knowledge. If you don't know the answer, say you are Sorry and unable to provide the information and provide only precise answer."""),
        ('human', '{question}')
    ])

    def format_docs(docs):
        return '\n\n'.join(doc.page_content for doc in docs)

    # Extract just the question string before passing to retriever
    chain = (
        {
            'context': RunnableLambda(lambda x: x['question']) | retriever | format_docs,
            'question': RunnableLambda(lambda x: x['question']),
        }
        | prompt
        | RunnableLambda(lambda x: llm.invoke(x))
        | StrOutputParser()
    )

    return chain.invoke({'question': question})

In [22]:
#print(rag_pipeline("What is the punishment for theft under Indian Penal Code?"))

print(rag_pipeline("Explain section 420 of indian penal code?"))

RETRIEVED CHUNKS:
[0] Section 498A of the Indian Penal Code or

[1] Indian Penal Code 1860
Section 1. Title and extent of operation of the Code
Act No. 45 of 1860.
This Act shall be called the Indian Penal Code, and sh

[2] Section 5. Certain laws not to be affected by this Act
15. Certain laws not to be affected by this Act.- Nothing in this Act shall affect 
the provisi

[3] Section 55A. Definition of appropriate Government
155A. Definition of “appropriate Government”.- In sections fifty-four and fifty-
five the expression

[4] understood subject to the exceptions contained in the Chapter entitled 
“General Exceptions”, though those exceptions are not repeated in such 
defini

[5] Non-bailable—Triable by Magistrate of the first class—Non-compoundable.
Para III
Punishment—Imprisonment for 3 years, or fine, or both—Cognizable—
Bai

[6] Section 40. Offence
140 “Offence”.- Except in the 2[Chapters] and sections mentioned in clauses 2 
and 3 of this section, the word “offence” denotes a



### Build the Gradio Interface for UI Interaction

In [ ]:
# Define the Gradio interface

rag_application = gr.Interface(
    fn = rag_pipeline,
    inputs = gr.Textbox(label='Question', lines=2, placeholder= 'Input your question related to Indian Penal Codes'),
    outputs = gr.Textbox(label='Answer'),
    title = "LawVerse",
    description="Your handy paralegal assistant!"
)

rag_application.launch(server_name="127.0.0.1", server_port= 7860,share=True)

* Running on local URL:  http://127.0.0.1:7860
* Running on public URL: https://4709438ee33d1d70ba.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
